<a href="https://colab.research.google.com/github/pySTEPS/ERAD-nowcasting-course-2026/blob/main/notebooks/exercise_notebooks/block_06_mlcast_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bonus: a nowcast on Belgian radar data from `mlcast-datasets`

The [mlcast-datasets](https://github.com/mlcast-community/mlcast-datasets) package of the
[MLCast community](https://github.com/mlcast-community) provides an intake catalog of open, cloud-hosted (zarr) radar datasets from several European services.

Here we grab a few frames of the Belgian RADCLIM 1 km / 5 min rain-rate product
(`be_rmi_radclim_mfb_5min`, RMI) straight from the cloud and run a plain pysteps
extrapolation nowcast on a summer convective case: 1 July 2020, around noon.

In [ ]:
!pip install -q mlcast-datasets pysteps
# Colab needs a restart-free reinstall of the zarr/xarray stack that the catalog uses
!pip install -q "zarr>=3" "xarray>=2024.10" s3fs

## 1. Open the catalog and load the Belgian data

In [ ]:
import mlcast_datasets

cat = mlcast_datasets.open_catalog()
print(list(cat.precipitation))

# 5-minute radar-gauge merged rain rate (mean-field-bias corrected), lazily via dask
ds = cat.precipitation.be_rmi_radclim_mfb_5min.to_dask()
ds

In [ ]:
# Select a window around noon on 1 July 2020 over Belgium
# and load it into memory (4 input frames + 12 verifying observations)
n_input, n_leadtimes = 4, 12
window = ds.rain_rate.sel(time=slice("2020-07-01T11:45", "2020-07-01T13:00")).load()
print(window.shape, window.time.values[0], "->", window.time.values[-1])

precip_input = window.isel(time=slice(0, n_input)).values      # mm/h
precip_obs = window.isel(time=slice(n_input, n_input + n_leadtimes)).values

## 2. Build the pysteps metadata

The grid is the Belgian Lambert 2008 projection (EPSG:3812), 1 km resolution, with `y` decreasing
(i.e. `yorigin="upper"`). pySTEPS only needs the corner coordinates and a proj4 string.

In [ ]:
import numpy as np
from pyproj import CRS

crs = CRS.from_wkt(ds[ds.rain_rate.grid_mapping].crs_wkt)
x, y = ds.x.values, ds.y.values
dx = float(x[1] - x[0])

metadata = {
    "projection": crs.to_proj4(),
    "x1": float(x[0] - dx / 2), "x2": float(x[-1] + dx / 2),
    "y1": float(y[-1] - dx / 2), "y2": float(y[0] + dx / 2),
    "xpixelsize": dx, "ypixelsize": dx,
    "yorigin": "upper",
    "unit": "mm/h", "transform": None, "accutime": 5.0,
    "threshold": 0.1, "zerovalue": 0.0,
}
metadata

## 3. Transform, estimate the motion field and nowcast

In [ ]:
from pysteps import motion, nowcasts
from pysteps.utils import transformation

# dB-transform the rain rates (NaNs -> zerovalue, as pysteps expects finite input)
precip_dbr, metadata_dbr = transformation.dB_transform(
    precip_input, metadata, threshold=0.1, zerovalue=-15.0
)
precip_dbr[~np.isfinite(precip_dbr)] = metadata_dbr["zerovalue"]

# Lucas-Kanade optical flow
velocity = motion.get_method("LK")(precip_dbr)

# Semi-Lagrangian extrapolation nowcast, 12 x 5 min = 1 hour ahead
nowcast_method = nowcasts.get_method("extrapolation")
precip_nowcast = nowcast_method(
    precip_dbr[-1], velocity, timesteps=n_leadtimes,
    extrap_kwargs={"allow_nonfinite_values": False},
)

# back-transform to rain rates
precip_nowcast = transformation.dB_transform(
    precip_nowcast, metadata_dbr, inverse=True
)[0]

## 4. Compare the nowcast with the observations

In [ ]:
import matplotlib.pyplot as plt
from pysteps.visualization import plot_precip_field

leadtimes = [2, 5, 8, 11]
fig, axes = plt.subplots(2, len(leadtimes), figsize=(16, 8))
for col, i in enumerate(leadtimes):
    for row, (field, label) in enumerate(
        [(precip_obs[i], "Observation"), (precip_nowcast[i], "Extrap. nowcast")]
    ):
        plt.sca(axes[row, col])
        plot_precip_field(field, geodata=metadata, colorscale="STEPS-NL", colorbar=False)
        plt.title(f"{label} +{(i + 1) * 5} min")
plt.tight_layout()

In [ ]:
from pysteps import verification

fss = verification.get_method("FSS")
scores = [fss(precip_nowcast[i], precip_obs[i], thr=1.0, scale=20) for i in range(n_leadtimes)]

plt.figure(figsize=(6, 4))
plt.plot(np.arange(1, n_leadtimes + 1) * 5, scores, "o-")
plt.xlabel("Lead time [min]"); plt.ylabel("FSS (thr=1 mm/h, scale=20 km)")
plt.grid(alpha=0.3)

## What next?

The catalog also holds German (`radklim_*`), Danish (`dmi_10_minutes`), Italian (`it_dpc_sri_5min`)
and UK (`uk_metoffice_5min`) radar data. Swap the catalog entry above, adapt the projection and
timestep in the metadata, and the same nowcasting code runs on any of them. Use it to test
how your method travels across radar domains!

*Data: RADCLIM, Royal Meteorological Institute of Belgium (CC-BY-4.0), served via the MLCast
community catalog on ECMWF object storage.*